In [39]:
import pandas as pd
import os
import kagglehub
from dotenv import load_dotenv
import numpy as np

In [2]:
#Load .env file for Kaggle Credentials 
load_dotenv()
username = os.getenv("KAGGLE_USERNAME")
key = os.getenv("KAGGLE_KEY")

os.environ["KAGGLE_USERNAME"] = username
os.environ["KAGGLE_KEY"] = key

In [3]:
#Locate path to the Kaggle dataset
path = kagglehub.dataset_download(
    "aiaiaidavid/the-big-dataset-of-ultra-marathon-running"
)

In [4]:
df = pd.read_csv(f"{path}/TWO_CENTURIES_OF_UM_RACES.csv", encoding="latin1")
df = df.drop(columns=["Event dates", "Event number of finishers", "Athlete club", "Athlete average speed"])

C:\Users\kimmi\AppData\Local\Temp\ipykernel_14920\1301004287.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{path}/TWO_CENTURIES_OF_UM_RACES.csv", encoding="latin1")


In [5]:
#1) Extract data since 2000s (TRY DIFFERENT THRESHOLDS)
df = df[df['Year of event'] >= 2000]

In [6]:
#2) Extract athletes that have data on a large portion of their career
# Get Athlete IDs where count exceeds 20 races (TRY DIFFERENT THRESHOLDS)
threshold_of_races = 20
valid_athletes = df['Athlete ID'].value_counts()
valid_athletes = valid_athletes[valid_athletes > threshold_of_races].index
experienced_df = df[df['Athlete ID'].isin(valid_athletes)]
experienced_df['Athlete ID'].nunique()

50969

In [11]:
#Feature of race_count --> Number of times a unique ID appears
experienced_df = df[df['Athlete ID'].isin(valid_athletes)].copy()
experienced_df["race_count"] = experienced_df.groupby("Athlete ID")["Athlete ID"].transform("count")

In [43]:
#Count the number of entries for each race
experienced_df['Event name'].value_counts()

Event name
Two Oceans Marathon (RSA)                                49684
Comrades Marathon - Down Run (RSA)                       34220
Comrades Marathon - Up Run (RSA)                         31416
Two Oceans Marathon - 50km Split (RSA)                   28242
Ultra Trail Tour du Mont Blanc (UTMB) (FRA)              11617
                                                         ...  
Open National Race Walking Championship Patiala (IND)        1
Rohring Round the Clock 6 hour Run (USA)                     1
Circuito Internacional de Marcha (MEX)                       1
Aarhus 1900 12 hours Indoor Run - 6h Split (DEN)             1
BÃ¸ Ultramarathon 45km (NOR)                                 1
Name: count, Length: 23142, dtype: int64

In [50]:
df_two_oceans = experienced_df[experienced_df['Event name'] == 'Two Oceans Marathon (RSA)'].copy()
df_two_oceans.dropna()
print(f"Total athletes: {df_two_oceans["Athlete ID"].nunique()}")
print(f"Total entries: {len(df_two_oceans["Athlete ID"])}")

Total athletes: 6721
Total entries: 49684


In [52]:
#Feature two_oceans_count --> Number of times they have ran this event
df_two_oceans["two_oceans_count"] = df_two_oceans.groupby("Athlete ID")["Athlete ID"].transform("count")
df_two_oceans["two_oceans_count"]

171456      5
171458      9
171463      9
171465      9
171466     10
           ..
6360604    11
6360613    12
6360630    16
6360638    13
6360639    14
Name: two_oceans_count, Length: 49684, dtype: int64

In [53]:
#Calculate the athletes average speed in km/hr --> Used to Extract top threshold of fastest runners
#Since format H:MM:SS h harder to compare, numerically

df_two_oceans["distance_km"] = df_two_oceans["Event distance/length"].str.extract(r"(\d+\.?\d*)").astype(float)
df_two_oceans["distance_km"]
time_parts = df_two_oceans["Athlete performance"].str.replace(" h", "").str.split(":", expand=True)

df_two_oceans["hours"] = (
    time_parts[0].astype(float) +
    time_parts[1].astype(float) / 60 +
    time_parts[2].astype(float) / 3600
)

df_two_oceans["calculated_speed"] = df_two_oceans["distance_km"] / df_two_oceans["hours"]
df_two_oceans["calculated_speed"].head()

171456    17.571690
171458    17.486339
171463    17.058724
171465    16.959704
171466    16.839292
Name: calculated_speed, dtype: float64

In [54]:
#Filter the top 5% of the fastest athletes
# adding feature for 'elite' runners, top 5% threshold can be adjusted
# df_two_oceans['Elite'] = df_two_oceans['Calculated speed'] >= df_two_oceans['Calculated speed'].quantile(0.95)  # per race or per year
threshold = df_two_oceans["calculated_speed"].quantile(0.90)

df_top10 = df_two_oceans[df_two_oceans["calculated_speed"] >= threshold].copy()

df_top10["finish_time_mins"] = df_top10["hours"] * 60

print(f"Total Entries: {len(df_two_oceans)}")
print(f"Top 10% Entries: {len(df_top10)}\n")
print(df_top10["calculated_speed"].describe())


Total Entries: 49684
Top 10% Entries: 4971

count    4971.000000
mean       13.815297
std         1.170941
min        12.426801
25%        12.848539
50%        13.552941
75%        14.404115
max        18.096948
Name: calculated_speed, dtype: float64


In [55]:
# runners are overwhelmingly from South Africa for this race
# potential domestic/foreign feature??
df_two_oceans["Athlete country"].value_counts()

Athlete country
RSA    47406
GBR      628
GER      287
NAM      206
ZIM      156
LES      111
USA       98
SWZ       87
AUS       70
NED       60
CAN       57
FRA       53
IRL       47
SUI       38
POR       35
BOT       34
ITA       33
SWE       29
BEL       29
RUS       25
MAW       19
CZE       19
IND       14
AUT       14
BLR       12
UGA       11
NZL       10
BRA        9
COD        8
HUN        7
ZAM        7
JPN        6
MOZ        6
NOR        6
ISR        6
TPE        5
POL        4
DEN        4
KEN        4
BEN        4
MAS        4
ANG        3
UAE        3
HKG        1
LUX        1
FIN        1
ARG        1
XXX        1
SVK        1
PHI        1
LAT        1
SGP        1
UKR        1
Name: count, dtype: int64

In [56]:
# 1. Athlete Age
df_top10["athlete_age"] = df_top10["Year of event"] - df_top10["Athlete year of birth"]

# 2. Age squared - captures non-linear age effect (performance peaks then declines)
df_top10["athlete_age_squared"] = df_top10["athlete_age"] ** 2

# 3. Gender - binary encode
df_top10["gender_male"] = (df_top10["Athlete gender"] == "M").astype(int)

# 4. Domestic vs International
df_top10["is_domestic"] = (df_top10["Athlete country"] == "RSA").astype(int)

# 5. Age from peak (marathon peak age ~28-32, using 30 as midpoint)
df_top10["age_from_peak"] = abs(df_top10["athlete_age"] - 30)

# 6. Experience - log transform to diminish returns of extra races
# log(1 + x) ensures athletes with 0 previous races don't cause log(0)
df_top10["experience_log"] = np.log1p(df_top10["race_count"])

# 7. Age x Gender interaction
df_top10["age_gender"] = df_top10["athlete_age"] * df_top10["gender_male"]

# 8. Age x log(experience) interaction
df_top10["age_x_experience"] = df_top10["athlete_age"] * df_top10["experience_log"]

In [58]:
features = [
    "finish_time_mins",
    "athlete_age",
    "athlete_age_squared",
    "gender_male",
    "is_domestic",
    "age_from_peak",
    "experience_log",
    "age_gender",
    "age_x_experience",
]

print(df_top10[features].corr())

                     finish_time_mins  athlete_age  athlete_age_squared  \
finish_time_mins             1.000000     0.245298             0.247153   
athlete_age                  0.245298     1.000000             0.992579   
athlete_age_squared          0.247153     0.992579             1.000000   
gender_male                 -0.031520     0.009808             0.013617   
is_domestic                  0.157659    -0.033042            -0.031060   
age_from_peak                0.247347     0.952863             0.970563   
experience_log               0.004875     0.109636             0.111146   
age_gender                   0.131572     0.655085             0.653418   
age_x_experience             0.212563     0.899973             0.895336   

                     gender_male  is_domestic  age_from_peak  experience_log  \
finish_time_mins       -0.031520     0.157659       0.247347        0.004875   
athlete_age             0.009808    -0.033042       0.952863        0.109636   
athlete_a